##### empty groupBy()

- is one **with no arguments** triggers a global aggregation across the **entire DataFrame**.
  - Similar to **SQL Group By** clause, It groups the **rows** of a DataFrame based on **one or more columns** and then applies an **aggregation function to each group**.
  - Common **aggregation functions** include **sum, count, mean, min, and max**.
    - `groupBy()` always needs an `aggregation`.
    - Use `.agg()` for `multiple calculations`.

**Syntax**

    groupBy(*cols)

    df.groupBy("column_name").agg(aggregation_function)
    df.groupBy(column_name1, column_name2, ...).agg(function1, function2, ...)

- Groups the DataFrame by the `specified columns` so that `aggregation` can be performed on them.

**cols:**
- `list, str, int or Column`

|        Scenario         |         Code Example                                    |            Result                                                   |
|-------------------------|---------------------------------------------------------|---------------------------------------------------------------------|
| empty groupby           | df.groupBy().max("").show()                             | Returns a 1-row DataFrame with the max salary of all employees.     |

In [0]:
from pyspark.sql.functions import sum

##### 1) Grouping an Empty DataFrame
- If the DataFrame itself has **no rows**, groupBy() will return an **empty result set (0 rows)**.

**a) retain the schema**
- If you perform a **groupBy** on a DataFrame that has **no rows**:
  - **Result:** The **resulting DataFrame** will also be **empty** but will **retain the schema (column names and types)** defined by the **aggregation**.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# columns = ["emp_name", "department", "state", "salary", "age", "bonus"]

schema = StructType([
    StructField("emp_name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("state", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("bonus", IntegerType(), True)
])

df_empty = spark.createDataFrame([], schema=schema)
display(df_empty)
df_empty.printSchema()

emp_name,department,state,salary,age,bonus


root
 |-- emp_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- state: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- bonus: integer (nullable = true)



In [0]:
from pyspark.sql.functions import sum

result_df = df_empty.groupBy("department") \
                    .agg(sum("salary").alias("total_salary"))

result_df.display()
result_df.printSchema()

department,total_salary


root
 |-- department: string (nullable = true)
 |-- total_salary: long (nullable = true)



- `No rows returned`.
- `Schema is preserved even though data is empty`.

**b) Efficiency Tip:**
- Use the `isEmpty()` method to check if a DataFrame `contains data` before running expensive operations like `groupBy`. 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# columns = ["emp_name", "department", "state", "salary", "age", "bonus"]

schema = StructType([
    StructField("emp_name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("state", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("bonus", IntegerType(), True)
])

df_empty = spark.createDataFrame([], schema=schema)
display(df_empty)

emp_name,department,state,salary,age,bonus


In [0]:
data = [(1, "HR", 5000),
        (2, "IT", 7000),
        (3, "HR", 6000),
        (4, "IT", 8000),
        (5, "Finance", 7500)]

columns = ["emp_id", "department", "salary"]

df_data = spark.createDataFrame(data, columns)
display(df_data)

emp_id,department,salary
1,HR,5000
2,IT,7000
3,HR,6000
4,IT,8000
5,Finance,7500


In [0]:
df_empty.groupBy("department").sum("salary").display()
df_data.groupBy("department").sum("salary").display()

department,sum(salary)


department,sum(salary)
HR,11000
IT,15000
Finance,7500


**Use isEmpty() Before groupBy**
- `groupBy = expensive shuffle operation`
- `If DataFrame is empty → waste of compute`
- `isEmpty() avoids unnecessary execution`

In [0]:
if not df_empty.isEmpty():
    result_df = df_empty.groupBy("department") \
                        .agg(sum("salary").alias("total_salary"))
    
    result_df.display()
else:
    print("DataFrame is empty. Skipping aggregation.")

DataFrame is empty. Skipping aggregation.


In [0]:
if not df_data.isEmpty():
    result_df = df_data.groupBy("department") \
                       .agg(sum("salary").alias("total_salary"))
    display(result_df)
else:
    print("DataFrame is empty. Skipping aggregation.")

department,total_salary
HR,11000
IT,15000
Finance,7500
